# LSTM Fine-tuning with Autoresearch Framework (Memory Optimized)

This notebook fine-tunes an LSTM model for next word prediction using concepts from [Karpathy's autoresearch](https://github.com/karpathy/autoresearch).

**Optimized for Colab GPU (T4 16GB)** - Uses mixed precision and gradient accumulation.

## 1. Setup and Installation

In [ ]:
# Clone autoresearch repo
!git clone https://github.com/karpathy/autoresearch.git -q 2>/dev/null || true

# Install dependencies
!pip install -q torch numpy matplotlib tiktoken requests

# Clear GPU memory
import torch, gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Import Libraries

In [ ]:
import os, math, time, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 3. Configuration (Optimized for Colab)

In [ ]:
# Sequence settings
MAX_SEQ_LEN = 128
TIME_BUDGET = 600

# Model hyperparameters (reduced for memory)
EMBEDDING_DIM = 128
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.2

# Training settings
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
LEARNING_RATE = 0.001
WARMUP_STEPS = 500
USE_AMP = True

print(f"Model params: embedding={EMBEDDING_DIM}, hidden={HIDDEN_DIM}, layers={NUM_LAYERS}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

## 4. Data Preparation

In [ ]:
# Sample training data
sample_texts = [
    "The quick brown fox jumps over the lazy dog. " * 50,
    "Machine learning is a subset of artificial intelligence. " * 50,
    "Deep learning uses neural networks with multiple layers. " * 50,
    "Natural language processing enables computers to understand human language. " * 50,
    "Long short term memory networks solve the vanishing gradient problem. " * 50,
    "Attention mechanisms have revolutionized sequence modeling tasks. " * 50,
    "Python is a popular programming language for machine learning. " * 50,
    "TensorFlow and PyTorch are leading deep learning frameworks. " * 50,
    "Neural networks learn patterns from data through backpropagation. " * 50,
    "Transformers have become the dominant architecture for NLP tasks. " * 50,
]
training_text = "\n".join(sample_texts)
print(f"Training text: {len(training_text):,} characters")

In [ ]:
import tiktoken

class BPETokenizer:
    def __init__(self):
        self.enc = tiktoken.get_encoding("cl100k_base")
    def encode(self, text):
        return [0] + self.enc.encode_ordinary(text)
    def decode(self, ids):
        return self.enc.decode(ids[1:] if ids and ids[0] == 0 else ids)
    def get_vocab_size(self):
        return self.enc.n_vocab

tokenizer = BPETokenizer()
vocab_size = tokenizer.get_vocab_size()
print(f"Vocab size: {vocab_size}")

In [ ]:
class TextDataset(Dataset):
    def __init__(self, text, tokenizer, max_length=MAX_SEQ_LEN):
        self.tokens = tokenizer.encode(text)
        self.sequences = []
        for i in range(0, len(self.tokens) - max_length, max_length):
            self.sequences.append(self.tokens[i:i + max_length + 1])
        print(f"Tokens: {len(self.tokens):,}, Sequences: {len(self.sequences):,}")
    def __len__(self):
        return len(self.sequences)
    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return torch.tensor(seq[:-1], dtype=torch.long), torch.tensor(seq[1:], dtype=torch.long)

dataset = TextDataset(training_text, tokenizer, MAX_SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
print(f"Batches per epoch: {len(dataloader)}")

## 5. LSTM Model Architecture

In [ ]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, 
                           dropout=dropout if num_layers > 1 else 0, batch_first=True)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.projection = nn.Linear(hidden_dim, embedding_dim)
        self.lm_head = nn.Linear(embedding_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None, reduction='mean'):
        x = self.dropout(self.embedding(idx))
        lstm_out, _ = self.lstm(x)
        x = self.layer_norm(lstm_out)
        x = F.relu(self.projection(x))
        logits = 15 * torch.tanh(self.lm_head(x) / 15)
        if targets is not None:
            return F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), 
                                  ignore_index=-1, reduction=reduction)
        return logits

model = LSTMLanguageModel(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. Evaluation (Bits Per Byte)

In [ ]:
@torch.no_grad()
def evaluate_bpb(model, dataloader, device, max_batches=20):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for i, (x, y) in enumerate(dataloader):
        if i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        loss = model(x, y, reduction='sum')
        total_loss += loss.item() * x.numel()
        total_tokens += x.numel()
    model.train()
    return (total_loss / total_tokens) / math.log(2) if total_tokens > 0 else float('inf')

print("Evaluation function ready")

## 7. Training Loop (Memory Optimized)

In [ ]:
class LRScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps=10000, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [g['lr'] for g in optimizer.param_groups]
        self.step_count = 0
    def step(self):
        self.step_count += 1
        if self.step_count < self.warmup_steps:
            mult = self.step_count / self.warmup_steps
        else:
            progress = min((self.step_count - self.warmup_steps) / (self.total_steps - self.warmup_steps), 1.0)
            mult = 0.5 * (1 + math.cos(math.pi * progress))
        for g, base in zip(self.optimizer.param_groups, self.base_lrs):
            g['lr'] = max(base * mult, self.min_lr)

print("LR Scheduler ready")

In [ ]:
def train_model(model, dataloader, device, time_budget=TIME_BUDGET):
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    scheduler = LRScheduler(optimizer, WARMUP_STEPS)
    
    history = {'loss': [], 'step': [], 'time': []}
    step, smooth_loss = 0, 0
    t_start = time.time()
    
    print(f"Training (budget: {time_budget}s, AMP: {USE_AMP}, GradAccum: {GRAD_ACCUM_STEPS})")
    print("=" * 70)
    
    while True:
        optimizer.zero_grad(set_to_none=True)
        for micro_step, (x, y) in enumerate(dataloader):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                loss = model(x, y) / GRAD_ACCUM_STEPS
            
            scaler.scale(loss).backward()
            
            if (micro_step + 1) % GRAD_ACCUM_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
                
                step += 1
                train_loss = loss.item() * GRAD_ACCUM_STEPS
                smooth_loss = 0.9 * smooth_loss + 0.1 * train_loss if smooth_loss else train_loss
                elapsed = time.time() - t_start
                
                if step % 10 == 0:
                    pct = min(elapsed / time_budget * 100, 100)
                    print(f"\rStep {step:4d} ({pct:4.1f}%) | Loss: {smooth_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f} | Time: {elapsed:.0f}s", end="")
                
                if step % 50 == 0:
                    history['loss'].append(smooth_loss)
                    history['step'].append(step)
                    history['time'].append(elapsed)
                
                if elapsed >= time_budget:
                    break
        
        if time.time() - t_start >= time_budget:
            break
    
    print(f"\n{'=' * 70}")
    print(f"Done! Steps: {step}, Time: {time.time() - t_start:.1f}s")
    return history

history = train_model(model, dataloader, device, time_budget=TIME_BUDGET)

## 8. Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['time'], history['loss'])
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Loss')
ax1.set_title('Loss vs Time')
ax1.grid(True, alpha=0.3)
ax2.plot(history['step'], history['loss'], color='orange')
ax2.set_xlabel('Steps')
ax2.set_ylabel('Loss')
ax2.set_title('Loss vs Steps')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final loss: {history['loss'][-1]:.4f}")
print(f"Total steps: {history['step'][-1]}")

In [ ]:
print("Evaluating...")
final_bpb = evaluate_bpb(model, dataloader, device)
print(f"Final BPB: {final_bpb:.4f}")

## 9. Text Generation

In [ ]:
def generate_text(model, tokenizer, prompt, max_length=50, temperature=0.7, top_k=40):
    model.eval()
    tokens = tokenizer.encode(prompt)
    input_ids = torch.tensor([tokens], dtype=torch.long, device=device)
    generated = list(tokens)
    
    with torch.no_grad():
        for _ in range(max_length):
            logits = model(input_ids)
            next_logits = logits[:, -1, :] / temperature
            if top_k > 0:
                top_vals, top_idx = torch.topk(next_logits, top_k)
                next_logits = torch.full_like(next_logits, float('-inf'))
                next_logits.scatter_(1, top_idx, top_vals)
            next_token = torch.multinomial(F.softmax(next_logits, dim=-1), 1)
            generated.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if input_ids.size(1) > MAX_SEQ_LEN:
                input_ids = input_ids[:, -MAX_SEQ_LEN:]
    
    model.train()
    return tokenizer.decode(generated)

print("=" * 60)
print("GENERATION EXAMPLES")
print("=" * 60)

for prompt in ["Machine learning", "Deep learning", "Neural networks", "Python is"]:
    text = generate_text(model, tokenizer, prompt, max_length=40)
    print(f"\n'{prompt}' -> {text[:150]}...")

In [ ]:
# Interactive generation
YOUR_PROMPT = "Deep learning"  # Change this
TEMPERATURE = 0.7               # 0.1-2.0

result = generate_text(model, tokenizer, YOUR_PROMPT, max_length=80, temperature=TEMPERATURE)
print(f"Prompt: '{YOUR_PROMPT}'")
print(f"Generated: {result}")

## 10. Save Model

In [ ]:
save_dir = '/content/lstm_model'
os.makedirs(save_dir, exist_ok=True)
torch.save(model.state_dict(), f'{save_dir}/model.pt')
config = {
    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'max_seq_len': MAX_SEQ_LEN,
    'final_bpb': final_bpb,
    'training_steps': history['step'][-1]
}
with open(f'{save_dir}/config.pkl', 'wb') as f:
    pickle.dump(config, f)
print(f"Model saved to {save_dir}")

## Summary

This notebook demonstrates:
1. **LSTM language model** inspired by autoresearch's architecture
2. **Memory optimization** via mixed precision and gradient accumulation
3. **Bits Per Byte (BPB)** evaluation metric
4. **Text generation** with temperature and top-k sampling

### Optimizations for Colab:
- Reduced model dimensions (128/256 vs 256/512)
- Mixed precision training (AMP)
- Gradient accumulation (effective batch 32 with micro-batch 8)
- Shorter sequences (128 vs 512)